# M2 Baseline — Text-to-SQL on Spider dev (Google Colab T4)

**Goal:** Benchmark 3 candidate models on Spider dev using execution accuracy (EX).

**Models tested:** Qwen2.5-Coder-7B-Instruct · defog/sqlcoder-7b-2 · OmniSQL-7B  
**Split:** Spider dev

**Workflow per model:**
1. Change `MODEL_ID` in section 6
2. Runtime → Restart and run all
3. Results auto-saved to Drive
4. Repeat for next model
5. Run section 11 to compare all 3


## 0. Runtime check — must show Tesla T4

In [1]:

!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
# If no GPU shown: Runtime > Change runtime type > T4 GPU > Save

Tesla T4, 15360 MiB


## 1. Install dependencies

In [2]:
!pip install -q transformers accelerate bitsandbytes sentencepiece python-dotenv sqlglot wandb pandas pyarrow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 MB 20.7 MB/s eta 0:00:00
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/base_command.py", line 179, in exc_logging_wrapper
    status = run_func(*args)
             ^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/req_command.py", line 67, in wrapper
    return func(self, options, args)
           ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/commands/install.py", line 447, in run
    conflicts = self._determine_conflicts(to_install)
                ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/commands/install.py", line 578, in _determine_conflicts
    return check_install_conflicts(to_install)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/operations/check.py", line 101, in check_install_co

## 2. Mount Google Drive

Your Spider data must be uploaded to Drive first. Expected layout:
```
MyDrive/text2sql_data/spider_data/dev.json
MyDrive/text2sql_data/spider_data/database/<db_id>/<db_id>.sqlite
```

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 3. Configure paths and clone repo

In [4]:
import os, sys

# ── EDIT THESE ────────────────────────────────────────────────────────────
GITHUB_TOKEN = ''   # GitHub PAT if repo is private; else leave empty
REPO_SLUG    = 'phuocNg964/text2sql-post-training'  # owner/repo
HF_TOKEN     = ''   # HuggingFace token (needed if model is gated)
WANDB_KEY    = ''   # Weights & Biases API key (optional)
# ──────────────────────────────────────────────────────────────────────────

REPO_DIR  = '/content/text2sql-post-training'
DATA_DIR  = '/content/drive/MyDrive/text-to-SQL-post-training'
PRED_DIR  = '/content/drive/MyDrive/text-to-SQL-post-training/predictions'
CACHE_DIR = '/content/drive/MyDrive/text-to-SQL-post-training/hf_cache'
for d in [PRED_DIR, CACHE_DIR]:
    os.makedirs(d, exist_ok=True)

if not os.path.exists(REPO_DIR):
    if GITHUB_TOKEN:
        clone_url = f'https://{GITHUB_TOKEN}@github.com/{REPO_SLUG}.git'
    else:
        clone_url = f'https://github.com/{REPO_SLUG}.git'
    os.system(f'git clone {clone_url} {REPO_DIR}')
else:
    os.system(f'git -C {REPO_DIR} pull')

sys.path.insert(0, REPO_DIR)
print('Repo ready:', REPO_DIR)


Repo ready: /content/text2sql-post-training


## 4. Authenticate (HuggingFace + W&B)

In [ ]:
from huggingface_hub import login
if HF_TOKEN:
    login(token=HF_TOKEN)
    print('HF authenticated')

import wandb
if WANDB_KEY:
    wandb.login(key=WANDB_KEY)
    print('W&B authenticated')
else:
    print('W&B skipped (no key)')

W&B skipped (no key)


## 5. Sanity check — gold SQL must give EX = 1.0

Run this once before any model. If it fails, there is a data path bug.

In [ ]:
from src.data.loader import load_spider
from src.eval.evaluator import evaluate

_records = load_spider(DATA_DIR, split='dev', n=20)
_gold    = [r['gold_sql'] for r in _records]
_result  = evaluate(_records, _gold)

print(f"Gold-SQL sanity check -> EX = {_result['execution_accuracy']}  (expected: 1.0)")
assert _result['execution_accuracy'] == 1.0, 'Data pipeline bug — fix before benchmarking!'
print('Sanity check passed')

Gold-SQL sanity check -> EX = 1.0  (expected: 1.0)
Sanity check passed


## 6. Select model

Change `MODEL_ID`, then **Runtime > Restart and run all** for the next model.

| # | Model | Size | Notes |
|---|---|---|---|
| 1 | `Qwen/Qwen2.5-Coder-1.5B-Instruct` | 1.5B | Small-scale boundary |
| 2 | `Qwen/Qwen2.5-Coder-3B-Instruct` | 3B | Primary compute-efficient candidate |
| 3 | `deepseek-ai/deepseek-coder-6.7b-instruct` | 6.7B | Cross-family peer |
| 4 | `Qwen/Qwen2.5-Coder-7B-Instruct` | 7B | Strong-scale candidate, SFT/RL target |

In [ ]:
# ── CHANGE THIS for each run ──────────────────────────────────────────────
MODEL_ID = 'Qwen/Qwen2.5-Coder-1.5B-Instruct'               # run 1
# MODEL_ID = 'Qwen/Qwen2.5-Coder-3B-Instruct'               # run 2
# MODEL_ID = 'deepseek-ai/deepseek-coder-6.7b-instruct'     # run 3
# MODEL_ID = 'Qwen/Qwen2.5-Coder-7B-Instruct'               # run 4

ENABLE_THINKING = False  # N/A for all four models above

N_SAMPLES = 100    # 50 ≈ 15-20 min; use 500 for the final baseline run
# ─────────────────────────────────────────────────────────────────────────

RUN_NAME = MODEL_ID.split('/')[-1].lower().replace('-', '_')
MODEL_DIR = os.path.join(PRED_DIR, RUN_NAME)
os.makedirs(MODEL_DIR, exist_ok=True)
OUT_FILE = os.path.join(MODEL_DIR, 'predictions.jsonl')
RES_FILE = os.path.join(MODEL_DIR, 'evaluation.json')
print(f'Model    : {MODEL_ID}')
print(f'Samples  : {N_SAMPLES}')
print(f'Dir      : {MODEL_DIR}')
print(f'Output   : {OUT_FILE}')

Model    : mistralai/Mistral-7B-Instruct-v0.3
Thinking : False
Samples  : 100
Output   : /content/drive/MyDrive/text-to-SQL-post-training/predictions/mistral_7b_instruct_v0.3_spider_dev.jsonl


## 8. Run inference

In [ ]:
N_SAMPLES

100

In [ ]:
# Section 8 in Colab — update inference cell
enable_flag = '--enable_thinking' if ENABLE_THINKING else ''
max_tokens = 1024 if ENABLE_THINKING else 256    # ← add this

!python {REPO_DIR}/scripts/run_inference.py \
    --model "{MODEL_ID}" \
    --split spider_dev \
    --data_dir "{DATA_DIR}" \
    --output "{OUT_FILE}" \
    --n_samples {N_SAMPLES} \
    --max_new_tokens {max_tokens} \
    {enable_flag}


  Model    : mistralai/Mistral-7B-Instruct-v0.3
  Split    : spider_dev
  Thinking : False
  Output   : /content/drive/MyDrive/text-to-SQL-post-training/predictions/mistral_7b_instruct_v0.3_spider_dev.jsonl
Loaded 100 records

Loading model...
config.json: 100% 601/601 [00:00<00:00, 3.07MB/s]
tokenizer_config.json: 100% 141k/141k [00:00<00:00, 147MB/s]
tokenizer.json: 100% 1.96M/1.96M [00:00<00:00, 56.8MB/s]

tokenizer.model: downloading bytes:   0% 0.00/587k [00:00<?, ?B/s]
tokenizer.model: downloading bytes: 100% 448k/448k [00:00<00:00, 703kB/s, 44.5kB/s  ]
tokenizer.model: reconstructing file: 100% 587k/587k [00:00<00:00, 919kB/s, 58.3kB/s  ]
special_tokens_map.json: 100% 414/414 [00:00<00:00, 2.94MB/s]
model.safetensors.index.json: 100% 23.9k/23.9k [00:00<00:00, 71.4MB/s]
Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 3 files:   0% 0/3 [00:00<?, ?it/s]
Reconstructing (incomplete total...):   0% 0.00/4.95G [00:00<?, ?B/s]         
Reconstruc

## 9. Evaluate

In [ ]:
import json
from src.data.loader import load_spider       # ← add this
from src.eval.evaluator import evaluate
from collections import defaultdict
records = load_spider(DATA_DIR, split='dev', n=N_SAMPLES)   # ← add this
predicted_sqls = [json.loads(l)['predicted_sql'] for l in open(OUT_FILE)]
result = evaluate(records, predicted_sqls)
exec_errors = sum(1 for r in result['results'] if r['execution_error'])

print('=' * 48)
print(f"  Model    : {MODEL_ID}")
print(f"  Split    : spider_dev")
print(f"  Samples  : {N_SAMPLES}")
print('-' * 48)
print(f"  EX       : {result['execution_accuracy']:.4f}")
print(f"  Correct  : {result['n_correct']} / {result['n_total']}")
print(f"  Exec err : {exec_errors} / {result['n_total']}")
print('=' * 48)

by_diff = defaultdict(lambda: {'correct': 0, 'total': 0})
for r in result['results']:
    d = r['difficulty'] or 'unknown'
    by_diff[d]['total'] += 1
    if r['correct']:
        by_diff[d]['correct'] += 1

print('\nBreakdown by difficulty:')
for d, v in sorted(by_diff.items()):
    ex = v['correct'] / v['total'] if v['total'] else 0
    print(f"  {d:<14}: {ex:.4f}  ({v['correct']}/{v['total']})")

with open(RES_FILE, 'w') as f:
    json.dump({'model': MODEL_ID, 'split': 'spider_dev', 'enable_thinking': ENABLE_THINKING, **result}, f, indent=2)
print(f'\nFull results -> {RES_FILE}')

  Model    : mistralai/Mistral-7B-Instruct-v0.3
  Split    : spider_dev
  Samples  : 100
------------------------------------------------
  EX       : 0.5500
  Correct  : 55 / 100
  Exec err : 19 / 100

Breakdown by difficulty:
  unknown       : 0.5500  (55/100)

Full results -> /content/drive/MyDrive/text-to-SQL-post-training/predictions/mistral_7b_instruct_v0.3_spider_dev_results.json


## 10. Log to Weights & Biases (optional)

In [ ]:
if WANDB_KEY:
    import wandb
    wandb.init(
        project='text2sql-post-training',
        name=f'm2-baseline-{RUN_NAME}-spider-dev',
        config={
            'model': MODEL_ID, 'split': 'spider_dev',
            'enable_thinking': ENABLE_THINKING, 'n_samples': N_SAMPLES,
            'quantization': 'nf4-4bit',
        },
    )
    wandb.log({
        'eval/spider_dev/execution_accuracy': result['execution_accuracy'],
        'eval/spider_dev/n_correct':          result['n_correct'],
        'eval/spider_dev/exec_errors':        exec_errors,
        **{f'eval/spider_dev/{d}_ex': v['correct']/v['total'] for d, v in by_diff.items()},
    })
    wandb.finish()
    print('Logged to W&B.')
else:
    print('W&B skipped (no key set).')

W&B skipped (no key set).


## 11. Compare all 3 models

Run after all model result files are saved to Drive. Picks the baseline.

In [ ]:
import json, os, glob

rows = []
for path in sorted(glob.glob(os.path.join(PRED_DIR, '*_results.json'))):
    with open(path) as f:
        d = json.load(f)
    rows.append({
        'model':   d['model'],
        'EX':      d['execution_accuracy'],
        'correct': d['n_correct'],
        'total':   d['n_total'],
        'errors':  sum(1 for r in d['results'] if r['execution_error']),
    })

rows.sort(key=lambda r: -r['EX'])
print(f"{'Model':<45} {'EX':>6}  {'Correct':>8}  {'Errors':>7}")
print('-' * 75)
for r in rows:
    print(f"{r['model']:<45} {r['EX']:>6.4f}  {r['correct']:>3}/{r['total']:<3}  {r['errors']:>7}")

if rows:
    print(f"\nBaseline candidate: {rows[0]['model']}")

Model                                             EX   Correct   Errors
---------------------------------------------------------------------------
Qwen/Qwen2.5-Coder-7B-Instruct                0.8000   16/20         1
Qwen/Qwen2.5-Coder-7B-Instruct                0.7700   77/100        2
Qwen/Qwen3-8B                                 0.6800   68/100        2
mistralai/Mistral-7B-Instruct-v0.3            0.5500   55/100       19

Baseline candidate: Qwen/Qwen2.5-Coder-7B-Instruct


### Push Predictions to GitHub
This cell will initialize the predictions directory as a git repository and push the results to your remote repo.

In [6]:
# Section 12 — Commit results to GitHub
import os, shutil, subprocess

YOUR_EMAIL = "nguyenquangphuoc5922@gmail.com"  # ← SỬA
YOUR_NAME  = "Phuoc Nguyen"                    # ← SỬA

PRED_DIR  = "/content/drive/MyDrive/text-to-SQL-post-training/predictions"
REPO_DIR  = "/content/text2sql-post-training"

def run(cmd):
    out = subprocess.check_output(cmd, shell=True, cwd=REPO_DIR,
                                  stderr=subprocess.STDOUT).decode()
    print(out)

pred_dest = os.path.join(REPO_DIR, "predictions")
os.makedirs(pred_dest, exist_ok=True)

for fname in os.listdir(PRED_DIR):
    if fname.endswith(".jsonl") or fname.endswith("_results.json"):
        shutil.copy(os.path.join(PRED_DIR, fname), os.path.join(pred_dest, fname))
        print(f"Copied: {fname}")

run(f"git config user.email '{YOUR_EMAIL}'")
run(f"git config user.name '{YOUR_NAME}'")
run("git add predictions/")
run(f"git commit -m 'M2: add baseline predictions'")
# Thay dòng run("git push") bằng:
run(f"git remote set-url origin https://{GITHUB_TOKEN}@github.com/{REPO_SLUG}.git")
run("git push")

print("Done.")


Copied: qwen3_8b_spider_dev.jsonl
Copied: qwen3_8b_spider_dev_results.json
Copied: mistral_7b_instruct_v0.3_spider_dev.jsonl
Copied: mistral_7b_instruct_v0.3_spider_dev_results.json
Copied: qwen2.5_coder_7b_instruct_spider_dev_results.json
Copied: qwen2.5_coder_7b_instruct_spider_dev.jsonl





CalledProcessError: Command 'git commit -m 'M2: add baseline predictions'' returned non-zero exit status 1.

In [8]:
import subprocess

REPO_DIR = "/content/text2sql-post-training"

result = subprocess.run(
    "git push", shell=True, cwd=REPO_DIR,
    capture_output=True, text=True
)
print("STDOUT:", result.stdout)
print("STDERR:", result.stderr)
print("Code:", result.returncode)


STDOUT: 
STDERR: fatal: could not read Username for 'https://github.com': No such device or address

Code: 128
